In [0]:
account = "olistdatastoragepradeep"
key = "key##"   

spark.conf.set(f"fs.azure.account.key.{account}.dfs.core.windows.net", key)


In [0]:
base_path  = "abfss://olistdata@olistdatastoragepradeep.dfs.core.windows.net"
silver_path = f"{base_path}/silver"
gold_path   = f"{base_path}/gold"

In [0]:
orders    = spark.read.format("delta").load(f"{silver_path}/orders")
items     = spark.read.format("delta").load(f"{silver_path}/order_items")
payments  = spark.read.format("delta").load(f"{silver_path}/payments")
customers = spark.read.format("delta").load(f"{silver_path}/customers")
products  = spark.read.format("delta").load(f"{silver_path}/products")
sellers   = spark.read.format("delta").load(f"{silver_path}/sellers")
reviews   = spark.read.format("delta").load(f"{silver_path}/reviews")

# optional
geo_zip   = spark.read.format("delta").load(f"{silver_path}/geolocation_zip")
prod_cat  = spark.read.format("delta").load(f"{silver_path}/product_categories")

 gold_fact_orders
 
Grain: 1 row per order
Rule: aggregate payments first (no row explosion)

In [0]:
from pyspark.sql.functions import col, sum as _sum

orders    = spark.read.format("delta").load(f"{silver_path}/orders")
payments  = spark.read.format("delta").load(f"{silver_path}/payments")
customers = spark.read.format("delta").load(f"{silver_path}/customers")

# payments -> 1 row per order
payments_agg = (payments
    .groupBy("order_id")
    .agg(_sum(col("payment_value")).alias("order_paid_amount"))
)

fact_orders = (orders
    .join(payments_agg, "order_id", "left")
    .join(customers.select("customer_id","customer_city","customer_state"), "customer_id", "left")
)

(fact_orders.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_orders"))



In [0]:
(fact_orders.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .save(f"{gold_path}/fact_orders"))

Gold Fact Table: Order Items

The fact_order_items table captures item-level sales transactions. It joins order items with product and seller information to analyze product-level revenue, freight costs, and seller performance.

In [0]:
from pyspark.sql.functions import col

items    = spark.read.format("delta").load(f"{silver_path}/order_items")
products = spark.read.format("delta").load(f"{silver_path}/products")
sellers  = spark.read.format("delta").load(f"{silver_path}/sellers")
orders   = spark.read.format("delta").load(f"{silver_path}/orders")

fact_order_items = (items.alias("i")
    .join(orders.select("order_id","customer_id","order_purchase_ts").alias("o"), "order_id", "left")
    .join(products.select("product_id","product_category_name").alias("p"),
          col("i.product_id")==col("p.product_id"), "left")
    .join(sellers.select("seller_id","seller_city","seller_state").alias("s"),
          col("i.seller_id")==col("s.seller_id"), "left")
    .select(
        col("i.order_id"), col("i.order_item_id"),
        col("o.customer_id"),
        col("i.product_id"), col("i.seller_id"),
        col("o.order_purchase_ts"),
        col("p.product_category_name"),
        col("s.seller_city"), col("s.seller_state"),
        col("i.price"), col("i.freight_value"),
        (col("i.price") + col("i.freight_value")).alias("item_total")
    )
)
fact_order_items.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_order_items")



In [0]:
(fact_order_items.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .save(f"{gold_path}/fact_order_items"))

In [0]:
dim_customers = customers.select("customer_id","customer_unique_id","customer_city","customer_state")
dim_customers.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(f"{gold_path}/dim_customers")

dim_products = products.select("product_id","product_category_name","product_weight_g","product_length_cm","product_height_cm","product_width_cm")
dim_products.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(f"{gold_path}/dim_products")

dim_sellers = sellers.select("seller_id","seller_city","seller_state")

dim_sellers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.dim_sellers")

In [0]:
dim_sellers.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(f"{gold_path}/dim_sellers")

Gold Mart: Revenue Monthly Analysis

This mart table provides aggregated revenue metrics by month and product category. It is created from the fact_order_items table and helps analyze sales trends, category performance, and revenue growth over time. This table is optimized for Power BI dashboards to visualize monthly revenue, number of orders, and freight contribution.

In [0]:
from pyspark.sql.functions import col, date_format, sum as _sum, countDistinct

fact_items = spark.read.format("delta").load(f"{gold_path}/fact_order_items")

mart_revenue_monthly = (fact_items
    .withColumn("year_month", date_format(col("order_purchase_ts"), "yyyy-MM"))
    .groupBy("year_month", "product_category_name")
    .agg(
        _sum(col("item_total")).alias("revenue"),
        _sum(col("price")).alias("product_revenue"),
        _sum(col("freight_value")).alias("freight_revenue"),
        countDistinct(col("order_id")).alias("orders")
    )
)
mart_revenue_monthly.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.mart_revenue_monthly")



In [0]:
(mart_revenue_monthly.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .save(f"{gold_path}/mart_revenue_monthly"))

 **Delivery Performance Analysis**

This mart table evaluates delivery efficiency and logistics performance by analyzing order timestamps from the fact_orders table. It measures delivery delays, average delivery time, and late delivery rate, helping identify operational issues in shipping and fulfillment.

In [0]:
from pyspark.sql.functions import col, to_date, avg, expr, count

fact_orders = spark.read.format("delta").load(f"{gold_path}/fact_orders")

mart_delivery_performance = (fact_orders
    .withColumn("purchase_date", to_date(col("order_purchase_ts")))
    .groupBy("purchase_date")
    .agg(
        count("*").alias("orders"),
        avg(col("delay_days")).alias("avg_delay_days"),
        avg(col("actual_delivery_days")).alias("avg_actual_delivery_days"),
        avg(col("estimated_delivery_days")).alias("avg_estimated_delivery_days"),
        expr("avg(case when delay_days > 0 then 1 else 0 end)").alias("late_rate")
    )
)

mart_delivery_performance.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.mart_delivery_performance")



In [0]:
(mart_delivery_performance.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .save(f"{gold_path}/mart_delivery_performance"))

In [0]:
spark.sql("SHOW TABLES IN gold").show()

In [0]:
spark.sql("SELECT COUNT(*) FROM gold.fact_orders").show()
spark.sql("SELECT COUNT(*) FROM gold.fact_order_items").show()
spark.sql("SELECT COUNT(*) FROM gold.mart_revenue_monthly").show()
spark.sql("SELECT COUNT(*) FROM gold.mart_delivery_performance").show()